# Synthetic HIV-recency dataset — guided demo

This notebook is a **self-contained, browser-friendly tour** of the synthetic
dataset that ships with the `Ukraine-HIV-Recency` pipeline. It is meant to run
either in a normal Jupyter kernel or in **JupyterLite** (Pyodide, in the browser)
so collaborators can explore the data without installing anything or touching
confidential records.

**What the dataset is.** A *fully synthetic* file (`synthetic_input_data.xlsx`)
with the exact schema the pipeline expects: a `hiv_cases` sheet and a
`testing_sites` sheet. It contains **no real record and no real coordinate**.
It is regenerated deterministically (`SEED=2026`) from non-personal,
site-level aggregates.

**How it is built (two steps, both reproducible).**
1. `extract_site_profiles.py` reads the real `data/input_data.xlsx` *locally* and
   writes only aggregate counts per site — overall (`site_profiles.csv`) and
   **per calendar month** (`site_monthly.csv`). Site identifiers are anonymised
   (`SITE_0001`, ...) and no coordinates leave the owner's machine.
2. `generate_synthetic.py` invents a site location as a random point inside the
   *correct oblast*, then "reseats" cases **month by month** using that month's
   recent/long-term and high/low-risk shares (with a small binomial jitter).

The month-by-month step is what lets the synthetic data reproduce the **temporal
trajectory** of the recent-infection share — and therefore the same hotspot
signal the detector looks for (current analysis window vs. historical baseline).

## Provenance & privacy

- **Synthetic, not de-identified.** Every row is generated; none is a real person.
- **Geography preserved only at the oblast level.** Exact clinic locations are
  hidden — a site sits at a random point within its true oblast, and each case is
  jittered ~2 km around its site.
- **Aggregates only.** The intermediate profile CSVs hold counts per site (and per
  site-month); they contain no individual records. Per-month counts are a finer
  granularity than per-site totals, so whether to publish `site_monthly.csv`
  alongside the dataset is the data owner's call.
- **Reproducible.** Re-running the two scripts with `SEED=2026` reproduces the
  same file bit-for-bit.

In [ ]:
# JupyterLite (in-browser) only: install the pure-Python deps this notebook needs.
# On a normal kernel these are already present and this cell is a no-op.
try:
    import piplite  # exists only inside JupyterLite / Pyodide
    await piplite.install(["pandas", "openpyxl", "matplotlib"])
except ImportError:
    pass

In [ ]:
from pathlib import Path
import pandas as pd

# Works whether the notebook is opened from synthetic_data/ or the project root.
HERE = Path.cwd()
XLSX = HERE / "synthetic_input_data.xlsx"
if not XLSX.exists():
    XLSX = HERE / "synthetic_data" / "synthetic_input_data.xlsx"

cases = pd.read_excel(XLSX, sheet_name="hiv_cases", parse_dates=["test_date"])
sites = pd.read_excel(XLSX, sheet_name="testing_sites",
                      parse_dates=["activation_date", "deactivation_date"])
print(f"Loaded {XLSX.name}")
print(f"  hiv_cases:     {len(cases):,} rows  {list(cases.columns)}")
print(f"  testing_sites: {len(sites):,} rows  {list(sites.columns)}")
cases.head()

In [ ]:
recent = cases["type"].str.lower().eq("recent")
high = cases["risk_group"].str.lower().eq("high")
summary = {
    "cases": len(cases),
    "sites_total": len(sites),
    "sites_with_cases": cases["site_id"].nunique(),
    "oblasts_covered": "see map below",
    "recent_share": round(recent.mean(), 4),
    "high_risk_share": round(high.mean(), 4),
    "date_min": cases["test_date"].min().date().isoformat(),
    "date_max": cases["test_date"].max().date().isoformat(),
}
pd.Series(summary, dtype=object)

## The temporal signal

The detector compares the recent-infection share in the **current analysis window**
against a **preceding baseline window** (the split is purely by `test_date`). The
synthetic generator reproduces the real share *per calendar month*, so this
current-vs-baseline contrast survives in the synthetic data. The plot below shows
the monthly recent share; months with very few tests are dropped to keep the line
readable.

In [ ]:
import matplotlib.pyplot as plt

m = cases.assign(rec=recent, ym=cases["test_date"].dt.to_period("M"))
monthly = m.groupby("ym")["rec"].agg(["sum", "count"])
monthly["share"] = monthly["sum"] / monthly["count"]
monthly = monthly[monthly["count"] >= 20]  # skip very sparse months

fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(monthly.index.to_timestamp(), monthly["share"], marker="o", ms=3)
ax.set_title("Synthetic monthly recent-infection share")
ax.set_ylabel("recent / all tested")
ax.set_xlabel("month")
ax.grid(alpha=0.3)
plt.show()

## Geography

Each synthetic site is placed at a random point inside its true oblast. The map
below needs `geopandas`; in a plain JupyterLite kernel that may be unavailable, so
the cell falls back to a simple longitude/latitude scatter. This is the natural
place to plug in **JupyterGIS** for an interactive map.

In [ ]:
try:
    import geopandas as gpd
    adm1 = HERE.parent / "data" / "Ukraine_Adm1_Oblast.geojson"
    if not adm1.exists():
        adm1 = HERE / "data" / "Ukraine_Adm1_Oblast.geojson"
    oblasts = gpd.read_file(adm1)
    gsites = gpd.GeoDataFrame(
        sites,
        geometry=gpd.points_from_xy(sites.longitude, sites.latitude),
        crs="EPSG:4326",
    )
    ax = oblasts.boundary.plot(figsize=(9, 7), color="0.7", linewidth=0.6)
    gsites.plot(ax=ax, color="#d62728", markersize=18)
    ax.set_title("Synthetic testing-site locations (random point within true oblast)")
    ax.set_axis_off()
    plt.show()
except Exception as exc:  # geopandas/geojson not available (e.g. plain JupyterLite)
    print("Map skipped, falling back to scatter:", exc)
    ax = sites.plot.scatter(x="longitude", y="latitude", s=12, figsize=(8, 6))
    ax.set_title("Synthetic testing-site locations")
    plt.show()

## Regenerating the dataset

From the project root, both steps are deterministic (`SEED=2026`):

```bash
python synthetic_data/extract_site_profiles.py   # reads the real data locally
python synthetic_data/generate_synthetic.py      # writes synthetic_input_data.xlsx
```

## Running the full pipeline on this dataset

This notebook intentionally stays lightweight (no PyMC) so it runs in the browser.
To run the **full Bayesian hotspot detection** on the synthetic file, point the
pipeline config's `excel_path` at `synthetic_data/synthetic_input_data.xlsx` and
run it from a full Python environment (PyMC is not available in JupyterLite):

```bash
python run_hotspots.py            # uses config.json (set excel_path to the synthetic file)
```